In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, concat, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType 

In [0]:
person_df = (
    spark.read
        .format("json")
        .load(f"{bronze_folder_path}/person.json")
)
display(person_df)

personId,personName
1,"List(George , Lucas)"
2,"List(Mark , Hamill)"
3,"List(Harrison , Ford)"
4,"List(Carrie , Fisher)"
5,"List(Peter , Cushing)"
6,"List(Anthony , Daniels)"
7,"List(Andrew , Stanton)"
8,"List(Lee , Unkrich)"
9,"List(Graham , Walters)"
10,"List(Bob , Peterson)"


In [0]:
(
    person_df.select(
        col("personId").alias("person_id"),
        col("personName.forename").alias("forename"),
        col("personName.surname").alias("surname")
    )
    .withColumn("fullname",
        concat(col("forename"), lit(" "), col("surname"))
    )
    .show()
)

+---------+----------+---------+------------------+
|person_id|  forename|  surname|          fullname|
+---------+----------+---------+------------------+
|        1|   George |    Lucas|     George  Lucas|
|        2|     Mark |   Hamill|      Mark  Hamill|
|        3| Harrison |     Ford|    Harrison  Ford|
|        4|   Carrie |   Fisher|    Carrie  Fisher|
|        5|    Peter |  Cushing|    Peter  Cushing|
|        6|  Anthony |  Daniels|  Anthony  Daniels|
|        7|   Andrew |  Stanton|   Andrew  Stanton|
|        8|      Lee |  Unkrich|      Lee  Unkrich|
|        9|   Graham |  Walters|   Graham  Walters|
|       10|      Bob | Peterson|     Bob  Peterson|
|       11|    David | Reynolds|   David  Reynolds|
|       12|Alexander |    Gould|  Alexander  Gould|
|       13|   Albert |   Brooks|    Albert  Brooks|
|       14|    Ellen |DeGeneres|  Ellen  DeGeneres|
|       18|     Brad |  Garrett|     Brad  Garrett|
|       19|  Allison |   Janney|   Allison  Janney|
|       20|E

In [0]:
personName_shema = StructType([
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])
person_schema = StructType([
    StructField("personId", IntegerType(), True),
    StructField("personName", personName_shema, True)
])
person_df = (
    spark.read
        .schema(person_schema)
        .format("json")
        .load(f"{bronze_folder_path}/person.json")
)
person_df.printSchema()
display(person_df)

root
 |-- personId: integer (nullable = true)
 |-- personName: struct (nullable = true)
 |    |-- forename: string (nullable = true)
 |    |-- surname: string (nullable = true)



personId,personName
1,"List(George , Lucas)"
2,"List(Mark , Hamill)"
3,"List(Harrison , Ford)"
4,"List(Carrie , Fisher)"
5,"List(Peter , Cushing)"
6,"List(Anthony , Daniels)"
7,"List(Andrew , Stanton)"
8,"List(Lee , Unkrich)"
9,"List(Graham , Walters)"
10,"List(Bob , Peterson)"


In [0]:
person_df = (
    add_ingestion_date(person_df)
        .select(
            col("personId").alias("person_id"),
            concat(col("personName.forename"), lit(" "), col("personName.surname")).alias("fullname"),
            lit(v_env).alias("env")
).drop("personName")
)

display(person_df)

person_id,fullname,env
1,George Lucas,dev
2,Mark Hamill,dev
3,Harrison Ford,dev
4,Carrie Fisher,dev
5,Peter Cushing,dev
6,Anthony Daniels,dev
7,Andrew Stanton,dev
8,Lee Unkrich,dev
9,Graham Walters,dev
10,Bob Peterson,dev


In [0]:
(
    person_df.write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/persons")
)


In [0]:
dbutils.notebook.exit("Success")